In [1]:
import pandas as pd

In [2]:
# Cargamos los datos

data = pd.read_csv("reddit_database_sentiment.csv", sep=";", engine="python")

In [3]:
data.head()

,created_date,created_timestamp,subreddit,title,author,author_created_utc,full_link,score,num_comments,num_crossposts,subreddit_subscribers,post,sentiment
0,2010-02-11 19:47:22,1265910442.0,analytics,So what do you guys all do related to analytic...,xtom,1.227476e+09,https://www.reddit.com/r/analytics/comments/b0...,7.0,4.0,0.0,NaN,There's a lot of reasons to want to know all t...,NEGATIVE
1,2010-03-04 20:17:26,1267726646.0,analytics,"Google's Invasive, non-Anonymized Ad Targeting...",xtom,1.227476e+09,https://www.reddit.com/r/analytics/comments/b9...,2.0,1.0,0.0,NaN,"I'm cross posting this from /r/cyberlaw, hopef...",NEGATIVE
2,2011-01-06 04:51:18,1294282278.0,analytics,"DotCed - Functional Web Analytics - Tagging, R...",dotced,1.294282e+09,https://www.reddit.com/r/analytics/comments/ew...,1.0,1.0,NaN,NaN,"DotCed,a Functional Analytics Consultant, offe...",NEGATIVE
3,2011-01-19 11:45:30,1295430330.0,analytics,Program Details - Data Analytics Course,iqrconsulting,1.288245e+09,https://www.reddit.com/r/analytics/comments/f5...,0.0,0.0,NaN,NaN,Here is the program details of the data analyt...,NEGATIVE
4,2011-01-19 21:52:28,1295466748.0,analytics,potential job in web analytics... need to anal...,therewontberiots,1.278672e+09,https://www.reddit.com/r/analytics/comments/f5...,2.0,4.0,NaN,NaN,i decided grad school (physics) was not for me...,POSITIVE


In [4]:
data.shape

(274239, 13)

In [5]:
data.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 274239 entries, 0 to 274238
Data columns (total 13 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   created_date           274238 non-null  object 
 1   created_timestamp      274215 non-null  object 
 2   subreddit              274209 non-null  object 
 3   title                  274209 non-null  object 
 4   author                 274209 non-null  object 
 5   author_created_utc     42585 non-null   float64
 6   full_link              274209 non-null  object 
 7   score                  274209 non-null  float64
 8   num_comments           274209 non-null  float64
 9   num_crossposts         224727 non-null  float64
 10  subreddit_subscribers  213890 non-null  float64
 11  post                   274209 non-null  object 
 12  sentiment              274203 non-null  object 
dtypes: float64(5), object(8)
memory usage: 405.7 MB


In [6]:
data.post

0         There's a lot of reasons to want to know all t...
1         I'm cross posting this from /r/cyberlaw, hopef...
2         DotCed,a Functional Analytics Consultant, offe...
3         Here is the program details of the data analyt...
4         i decided grad school (physics) was not for me...
                                ...                        
274234    Hello! I am wonder how the following output wo...
274235    Can anybody recommend me a book with medical s...
274236    I'm trying to build a simple Markov chain. I h...
274237    How do I view all available Rcpp::plugins? Tha...
274238    Hi everybody,\n\nI am currently doing a factor...
Name: post, Length: 274239, dtype: object

In [7]:
# Cantidad de valores faltantes

data.post.isna().sum()

30

In [8]:
# Análisis general de los posts

data.post.tolist()

["There's a lot of reasons to want to know all this stuff, so I figured I'd get to know the others that are on this subreddit.\n\nSo let's hear it: Webmasters? Coders? Marketers? Work for an analytics software company? You get the idea.",
 'I\'m cross posting this from /r/cyberlaw, hopefully you guys find it as interesting as I did(it deals with Google Analytics):\n\nSo quite awhile ago, I ordered a Papa John\'s pizza online. My job largely involves looking at ads that appear online, so afterwards I was quick to notice *I was getting a LOT* of Papa Johns ads (especially at night) being served through a Google owned company (DoubleClick media). Yesterday one of these ads popped up again on Youtube (a place that typically serves using the adwords program, not doubleclick), so I decided to copy the URL. \n\nFor those not in the advertising field: Making full use of Google\'s analytics tool means that certain information about the advertising campaign is leaked in the URL.\n\nSo let\'s bre

In [9]:
data.post.str.contains("<")

0         False
1         False
2         False
3         False
4         False
          ...  
274234    False
274235    False
274236    False
274237    False
274238    False
Name: post, Length: 274239, dtype: object

In [10]:
# Rellemos los valores faltantes con una oración por defecto

data.post.fillna("Sin contenido", inplace=True)

In [11]:
# Comprobación de etiquetas HTML

data.post[data.post.str.contains("<")].tolist()

['I have just recently begun studying Deep Learning and also have no previous experience with TensorFlow. I am very confident in my understanding of the theoretical side of Neural Networks and have written a couple of very simple RNNs from scratch. I have now started to become familiar with using TensorFlow and have been trying to implement slightly more advanced NNs using it.\r\n\r\nCurrently, I am writing an LSTM that reads sequences of characters from a text, character-by-character, and then (hopefully) learns to accurately predict the next character. I encoded each character in the vocab as a one-hot vector of length equal to the size of the vocab, and the 1 represents the key to that character in my `int : char` map. The model takes as input `n` batches (100 in the example) of `m` character input and label sequences (25 in the example). These input sequences are just any random sequence of `m` characters from the first 80% of the text, the labels are the same sequences, but shifte

In [12]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import string
import re
import contractions
import emoji

# Descargamos los recursos necesarios de NLTK
nltk.download("punkt", quiet=True)   # Tokenización
nltk.download("stopwords", quiet=True)   # Stopwords

# Definimos las expresión regulares
url_pattern = re.compile(r"http[s]?://\S+")
html_pattern = re.compile(r"<.*?>")

# Creamos un conjunto de stopwords en inglés para filtrar palabras comunes
stop_words = set(stopwords.words("english"))


# Función para convertir emojis a texto descriptivo
def convert_emojis_to_text(text):
    return emoji.demojize(text)


def preprocess_post(text: str):
    # Paso 1: Convertir todo el texto a minúsculas y resolver contracciones (como "I'm" a "I am")
    text = contractions.fix(text.lower())

    # Paso 2: Eliminar signos de puntuación y espacios extra
    text = text.translate(str.maketrans("", "", string.punctuation)).strip()

    # Paso 3: Eliminar URLs
    text = url_pattern.sub("", text)
    
    # Paso 4: Eliminar etiquetas HTML
    text = html_pattern.sub("", text)

    # Paso 5: Convertir los emojis en el texto a su representación textual (ej. ":smile:" en lugar de "😊")
    text = convert_emojis_to_text(text)

    # Paso 6: Tokenizar el texto, eliminar palabras que no sean alfabéticas y que sean stopwords
    sin_stopwords = [word for word in word_tokenize(text) if word.isalpha() and word not in stop_words]

    # Paso 7: Unir las palabras procesadas en un único string, separado por espacios
    clean_text = " ".join(sin_stopwords)

    return clean_text   # Devolver el texto limpio y preprocesado

In [13]:
# Aplicamos la función de preprocesado y normalización a cada uno de los posts y almacenos los resultados en una nueva columna llamada 'clean_post'

data["clean_post"] = data.post.apply(preprocess_post)

In [14]:
data.head()

,created_date,created_timestamp,subreddit,title,author,author_created_utc,full_link,score,num_comments,num_crossposts,subreddit_subscribers,post,sentiment,clean_post
0,2010-02-11 19:47:22,1265910442.0,analytics,So what do you guys all do related to analytic...,xtom,1.227476e+09,https://www.reddit.com/r/analytics/comments/b0...,7.0,4.0,0.0,NaN,There's a lot of reasons to want to know all t...,NEGATIVE,lot reasons want know stuff figured would get ...
1,2010-03-04 20:17:26,1267726646.0,analytics,"Google's Invasive, non-Anonymized Ad Targeting...",xtom,1.227476e+09,https://www.reddit.com/r/analytics/comments/b9...,2.0,1.0,0.0,NaN,"I'm cross posting this from /r/cyberlaw, hopef...",NEGATIVE,cross posting rcyberlaw hopefully guys find in...
2,2011-01-06 04:51:18,1294282278.0,analytics,"DotCed - Functional Web Analytics - Tagging, R...",dotced,1.294282e+09,https://www.reddit.com/r/analytics/comments/ew...,1.0,1.0,NaN,NaN,"DotCed,a Functional Analytics Consultant, offe...",NEGATIVE,dotceda functional analytics consultant offeri...
3,2011-01-19 11:45:30,1295430330.0,analytics,Program Details - Data Analytics Course,iqrconsulting,1.288245e+09,https://www.reddit.com/r/analytics/comments/f5...,0.0,0.0,NaN,NaN,Here is the program details of the data analyt...,NEGATIVE,program details data analytics certification c...
4,2011-01-19 21:52:28,1295466748.0,analytics,potential job in web analytics... need to anal...,therewontberiots,1.278672e+09,https://www.reddit.com/r/analytics/comments/f5...,2.0,4.0,NaN,NaN,i decided grad school (physics) was not for me...,POSITIVE,decided grad school physics branching job mark...


In [15]:
# data.to_csv("processed_dataset.csv")